In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import Dataset as ds
import random
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import pandas as pd
import gc

# Configurar semillas para facilitar la reproducibilidad de los resultados
seed = 44
torch.manual_seed(seed)
random.seed(seed)
set_seed(seed)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [ ]:
import transformers
print(transformers.__version__)

4.57.1


In [ ]:
from huggingface_hub import login
TOKEN = ''
login(token=TOKEN)

In [ ]:
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-1.7B-Base',device_map='auto',torch_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B-Base", trust_remote_code=True)
tokenizer.add_special_tokens({'pad_token': '<|image_pad|>'})
model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
tokenizer.pad_token_id

151655

In [ ]:
dtypes = {param.dtype for param in model.parameters()}
dtypes

{torch.float16}

Se cargan los datos de entrenamiento.

In [ ]:
df = pd.read_excel('train.xlsx')
ds_train = ds.from_pandas(df)
ds_train = ds_train.train_test_split(test_size=0.05)

In [ ]:
ds_train

DatasetDict({
    train: Dataset({
        features: ['codigo_comun', 'non_pls', 'pls'],
        num_rows: 3384
    })
    test: Dataset({
        features: ['codigo_comun', 'non_pls', 'pls'],
        num_rows: 179
    })
})

In [ ]:
base_prompt = """Using the next instructions, write a plain-language summary for patients:
* Organize it using short, reader-friendly headings similar to those used in patient-oriented evidence summaries (e.g., Review question, Background, Study characteristics, Key results, Conclusions, Quality of evidence).
* Do not use the technical headings verbatim; instead, adapt them into simple, descriptive labels.
* Integrate all information into a clear, coherent, and easy-to-follow narrative.
* Keep the language at or below a 6th-grade reading level.
* Avoid jargon; if you must use a technical term, explain it in simple, familiar words.
* Use active voice, mostly short words (one or two syllables), and sentences of no more than 20 words.
* Organize the summary into short paragraphs of 3-5 sentences
* Use simple numbers or ratios (for example, 1 in 2) instead of percentages.
* Do not add, remove, or infer any information not present in the abstract.
* The summary should remain consistent regardless of the order of sentences in the original abstract.
* Provide only the summary as plain text. Do not use bold, italics, headings, asterisks, or any other formatting
* Do not provide comments or explanations.
* Target a total length of 300–500 words.
Here is the abstract of a biomedical study to summarize:"""

In [ ]:
def format_sample(sample, base_prompt):
    '''
    Función para crear prompt a partir de un texto técnico
    '''
    non_pls = sample["non_pls"]

    prompt = (
        f"{base_prompt}\n\n"
        f"{non_pls}\n\n"
        "Plain-language summary:\n\n"
    )
    return prompt

In [ ]:
class QADataset(Dataset):
    def __init__(self, dataset, tokenizer, base_prompt):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.base_prompt = base_prompt

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        non_pls = sample['non_pls']
        pls = sample['pls']

        prompt = format_sample({'non_pls': non_pls}, self.base_prompt)

        prompt_enc = self.tokenizer(prompt, truncation=True)
        prompt_len = len(prompt_enc['input_ids'])
        full_prompt = prompt + f"{pls}" + '<|endoftext|>'
        full_enc = self.tokenizer(full_prompt, truncation=True)

        input_ids = full_enc['input_ids']
        attention_mask = full_enc['attention_mask']
        labels = [-100] * len(input_ids)
        labels[prompt_len:] = input_ids[prompt_len:]

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

In [ ]:
train_data = QADataset(ds_train['train'], tokenizer, base_prompt)
eval_data = QADataset(ds_train['test'], tokenizer, base_prompt)

In [ ]:
def custom_collate_fn(batch):
    max_length = max([len(x["input_ids"]) for x in batch])

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:

        padded_input_ids = item["input_ids"] + [tokenizer.pad_token_id] * (max_length - len(item["input_ids"]))
        input_ids.append(padded_input_ids)

        padded_attention_mask = item["attention_mask"] + [0] * (max_length - len(item["attention_mask"]))
        attention_masks.append(padded_attention_mask)

        padded_labels = item["labels"] + [-100] * (max_length - len(item["labels"]))
        labels.append(padded_labels)

    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_masks),
        "labels": torch.tensor(labels)
    }

In [ ]:
config = LoraConfig(
    r=16, # Rank de las matrices A y B
    lora_alpha=16, # Factor de regularización de las matrices A y B
    target_modules=["q_proj", 'k_proj', 'v_proj', 'o_proj'], # Nombre de las capas lineales a las que se les aplicará LoRA
    lora_dropout=0.05, # Dropout de las matrices A y B
    bias="none", # No se añade bias a las capas lineales
    task_type="CAUSAL_LM" # Tipo de tarea
)

# Se obtiene el modelo con LoRA
model = get_peft_model(model, config)

In [ ]:
trainer = Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        per_device_eval_batch_size=1,
        warmup_ratio=0.03,
        # max_grad_norm=1,
        # gradient_checkpointing=True,
        # num_train_epochs=2,
        learning_rate=1e-4,
        # bf16=True,
        fp16=True, # Usar precisión de 16 bits
        eval_strategy='steps',
        max_steps=1800,
        eval_steps=50,
        logging_steps=50,
        # torch_empty_cache_steps=2,
        save_strategy='steps',
        save_steps=50,####
        eval_accumulation_steps=1,
        seed=seed,
        data_seed=seed,
        disable_tqdm=False,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1
    ),
    data_collator=custom_collate_fn
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
gc.collect()

451

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
50,1.469900,1.439708
100,1.428700,1.413158
150,1.411300,1.402865
200,1.366700,1.396694
250,1.365700,1.392977
300,1.363800,1.388974
350,1.360700,1.385140
400,1.389300,1.382522
450,1.379800,1.379889
500,1.345900,1.378022


TrainOutput(global_step=1800, training_loss=1.3437859725952148, metrics={'train_runtime': 4099.4245, 'train_samples_per_second': 3.513, 'train_steps_per_second': 0.439, 'total_flos': 2.469794797548933e+17, 'train_loss': 1.3437859725952148, 'epoch': 4.25531914893617})

In [ ]:
save_dir = "Qwen3-1.7B-01"
model.save_pretrained(save_dir)